# Demo: Training a Neural Forecasting Model

We have monthly subscriber counts for a streaming service -- 5 years of data, classic S-curve growth. ARIMA is going to struggle here because it assumes linear dynamics. Let's train an N-BEATS model and see if a neural network can do better.

In [ ]:
import os
import warnings

os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.utils.likelihood_models import QuantileRegression

HORIZON = 12

## Load the Data

Monthly active subscribers from Jan 2020 to Dec 2024. Let's plot it and see what we're working with.

In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date").asfreq("MS")
subscribers = df["subscribers"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(subscribers.index, subscribers.values, color="black", linewidth=1.5)
ax.set_ylabel("Subscribers")
ax.set_title("Monthly Active Subscribers")
ax.axvline(pd.Timestamp("2024-01-01"), color="red", linestyle="--", alpha=0.5, label="Train/Test split")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Data: {len(subscribers)} months, {subscribers.index[0].date()} to {subscribers.index[-1].date()}")

See the S-curve? Early growth is explosive -- the service is new, everyone's signing up. Then it decelerates as the market saturates. ARIMA assumes linear dynamics, so it's going to struggle with this shape. It'll basically draw a straight line and call it a day.

## Prepare the Data

Darts wants its own `TimeSeries` object. We split at end of 2023 -- train on 4 years, test on the final 12 months.

In [ ]:
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))

print(f"Train: {len(train)} months")
print(f"Test:  {len(test)} months")

## Train N-BEATS with Quantile Regression

N-BEATS is a pure deep-learning forecaster -- stacks of fully connected layers, no recurrence. Let's see what it can do with this S-curve data.

The key parameters:
- **`input_chunk_length=24`** -- it looks at 2 years of history to make each prediction.
- **`output_chunk_length=12`** -- it predicts 12 months ahead in one shot.
- **`QuantileRegression`** -- instead of just a point forecast, we get uncertainty bands. The quantiles (0.05, 0.25, 0.5, 0.75, 0.95) give us a 90% prediction interval plus a tighter 50% interval.

This will take a minute to train.

In [ ]:
nbeats = NBEATSModel(
    input_chunk_length=24,
    output_chunk_length=12,
    n_epochs=50,
    likelihood=QuantileRegression(quantiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True},
)
nbeats.fit(train)
print("N-BEATS training complete.")

Now generate the forecast. `num_samples=100` draws from the learned quantile distribution to give us probabilistic predictions.

In [ ]:
nbeats_fc = nbeats.predict(HORIZON, num_samples=100)

## ARIMA Baseline

Let's fit a simple ARIMA(2,1,1) so we have something to compare against. This is the same model we used in Lesson 3 -- it works great for linear trends, not so great for curves.

In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)
print("ARIMA fit complete.")

## Compare the Forecasts

This is the payoff. Let's plot both forecasts against the actual 2024 data.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Plot actuals
ts.plot(ax=ax, label="Actual", color="black", linewidth=1.5)

# ARIMA forecast
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color="tab:blue")

# N-BEATS forecast with uncertainty bands
nbeats_fc.plot(ax=ax, label="N-BEATS", color="tab:orange",
               low_quantile=0.05, high_quantile=0.95)

ax.set_ylabel("Subscribers")
ax.set_title("N-BEATS vs ARIMA -- Subscriber Forecast")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

N-BEATS captures the curve. ARIMA projects a straight line. This is the difference between a model that can learn nonlinear patterns and one that can't.

The orange shaded band is the 90% prediction interval from the quantile regression. It tells us not just *where* the forecast is, but *how uncertain* the model is about it.

In [ ]:
print("=== Forecast at Month 12 ===")
print(f"  Actual:  {test.values().flatten()[-1]:,.0f}")
print(f"  ARIMA:   {arima_fc.values().flatten()[-1]:,.0f}")
print(f"  N-BEATS: {nbeats_fc.values().flatten()[-1]:,.0f}")